# Inference Pipeline

In [1]:
# install the package
# !pip install --upgrade setuptools packaging
# !pip install -e ..

In [2]:
# Install TA-Lib using conda
# !conda install -c conda-forge ta-lib -y

In [3]:
!nvidia-smi

Thu Aug 29 14:16:48 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 546.80                 Driver Version: 546.80       CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8               2W /  82W |    406MiB /  8188MiB |      3%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import torch

import warnings

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.options.utils.data_handler import DataProcessor, InferenceDataHandler
from blockhouse_ml.options.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.options.utils import fetch_merge_data
from blockhouse_ml.options.utils.fetch_merge_data import PolygonClient
from blockhouse_ml.options.utils.env import TradingEnvironment

c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-08-29 14:16:56,074	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-08-29 14:16:56,209	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [5]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
'''
Because of this error:
 [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.
'''

'\nBecause of this error:\n [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.\nOMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.\n'

In [6]:
warnings.filterwarnings("ignore")

In [7]:
## Initialize the variables
# Initialize the model directory, where the models will be saved
MACRO_MODEL_DIR = '../OptionsTabModels'
os.makedirs(MACRO_MODEL_DIR, exist_ok=True)

# MICRO_MODEL_DIR = 'MicroModels'
# os.makedirs(MICRO_MODEL_DIR, exist_ok=True)


# Initialize the data directory, where the data will be stored
data_dir = '../OptionsData'
os.makedirs(data_dir, exist_ok=True)

# Initialize the MacroTraderModel
macro_trader = MacroTraderModel(MACRO_MODEL_DIR)

# Initialize the MicroTraderModel
# micro_trader = MicroTraderModel(MICRO_MODEL_DIR)

# Initialize the data client
data_client = PolygonClient(data_dir)

# Initialize the data processor
data_processor = DataProcessor()

# Inference data handler
inference_data_handler = InferenceDataHandler()

# Initialize the MetaLearner
meta = MetaLearner()


In [8]:
# For getting the data in real time 
ticker = 'AAPL'

user_data = {
    "maturity_date": "240830",
    "option_type": "C",
    "strike_price": 100
}

# Get today's date
end_date = datetime.strptime(user_data["maturity_date"], "%y%m%d")-timedelta(days=1)
# Calculate the start date (7 days before today's date)
start_date = (end_date - timedelta(days=10)).strftime('%Y-%m-%d')
end_date = end_date.strftime('%Y-%m-%d')

timeframe=500
inventory=10000

print("LOGGING: Starting pipeline for", ticker, "from", start_date, "to", end_date)



LOGGING: Starting pipeline for AAPL from 2024-08-19 to 2024-08-29


In [9]:
def get_schedule(timeframe, transaction_size, market_cap_int, input_row, data,get_tab_transformer = False):
    """
    Generates a trading schedule based on the transaction size and input data.

    Args:
    - timeframe (int): The timeframe for trade execution.
    - transaction_size (int): The size of the transaction.
    - input_row (pd.DataFrame): The input data row with forecasts and technical indicators.

    Returns:
    - list: A list of trades executed based on the generated schedule.
    """
    print("LOGGING: Generating Schedule...")
    # print("input_row", input_row.columns)
    trades, micro_input = macro_trader.infer_macro(timeframe, transaction_size, market_cap_int,  input_row, data, meta=meta, inference_data_handler=inference_data_handler,get_tab_transformer=get_tab_transformer)
    return trades, micro_input

# Start inferencing

In [10]:
def run_pipeline(ticker, start_timestamp, end_timestamp, timeframe, inventory, user_data, trade_set_counter=1, get_tab_transformer = False):
    """
    Runs the entire pipeline for the trading model, including data retrieval, 
    technical indicator addition, forecasting, and generating trade schedules.

    Returns:
    - list: The list of trades generated by the model.
    """
    start_time = time.time()
    # Create the trading environment
    data_filepath = f'{data_dir}/inference_merged_appl_{start_timestamp}_{end_timestamp}.csv'
    if os.path.exists(data_filepath):
        data = pd.read_csv(data_filepath)
    else:
        data = data_client.fetch_and_merge_data(ticker,start_date=start_timestamp,end_date=end_timestamp, maturity_date=user_data['maturity_date'], option_type=user_data['option_type'],strike_price=user_data['strike_price'])
        data.to_csv(f'{data_dir}/inference_merged_appl_{start_timestamp}_{end_timestamp}.csv')
    data = data_processor.technical_indicators.add_technical_indicators(data,option_type=user_data['option_type'],strike_price=user_data['strike_price'])
    input_row = inference_data_handler.add_forecast(data)
    market_cap_int = data_client.get_market_cap(ticker)
    
    # trades, micro_input = get_schedule(timeframe, inventory, market_cap_int, input_row, data, get_tab_transformer=get_tab_transformer)
    return trades, micro_input
    # micro_input['Timestamp'] = trades['timestamp'].tolist()
    # # Display the column names of the trades DataFrame
    # # print(micro_input)
    # # macro_timestamps = trades['timestamp'].tolist()
    # micro_input['Ticker'] = [ticker]*len(micro_input)
    # micro_input['Inventory'] = [inventory]*len(micro_input)
    # micro_input['Trade_Set_ID'] = [f"set_{trade_set_counter}"]*len(micro_input)


    # # ## run micro trader
    # micro_env = TradingEnvironmentMicro(micro_input, preferred_timeframe=timeframe, initial_inventory=inventory)
    # micro_model = micro_trader.load_model(micro_env, {}) 
 
    # # Reset the environment to start inference
    # obs = micro_env.reset()

    # # Initialize an empty list to store trade details
    # trade_details = []

    # for _ in range(len(micro_input)):
    #     # Predict the action to take based on the current observation
    #     action, _states = micro_model.predict(obs)
        
    #     # Step through the micro_environment using the predicted action
    #     obs, rewards, done, info = micro_env.step(action)
        
    #     # Extract and print the trade details
    #     order_type = "Market" if action[0] < 0.5 else "Limit"
    #     execution_price = action[1] if order_type == "Limit" else obs[3]  # Assuming 'close' price is at index 3
        
    #     # Append the trade details to the list
    #     trade_details.append({
    #         'Order Type': order_type,
    #         'Limit Price': execution_price
    #     })
        
    #     # print(f"Trade Executed: {order_type} Order at Price: {execution_price}")
        
    #     # If the micro_environment is done, break the loop
    #     if done:
    #         break

    # # Render the final state of the micro_environment
    # micro_env.render()
    
    # return trade_details

In [11]:
import warnings

# Suppress the specific ConvergenceWarning
warnings.filterwarnings("ignore", message="The optimizer returned code 4. The message is:\nInequality constraints incompatible")

# Your code that generates the warning


In [12]:
# import warnings

# # Ignore all warnings
# warnings.filterwarnings("ignore")

import warnings

# Suppress the specific ConvergenceWarning
warnings.filterwarnings("ignore", message="The optimizer returned code 4. The message is:\nInequality constraints incompatible")

trade_details = run_pipeline(ticker, start_date, end_date, timeframe, inventory, user_data,get_tab_transformer=True)

getting quotes -->  2024-08-19 2024-08-29


100%|██████████| 11/11 [01:51<00:00, 10.16s/it]


Skipping row 2024-08-29 13:00:00 due to missing or invalid data.
Skipping row 2024-08-29 13:25:00 due to missing or invalid data.
Skipping row 2024-08-29 13:30:00 due to missing or invalid data.
Skipping row 2024-08-29 13:31:00 due to missing or invalid data.
Skipping row 2024-08-29 13:33:00 due to missing or invalid data.
Skipping row 2024-08-29 13:34:00 due to missing or invalid data.
Skipping row 2024-08-29 13:35:00 due to missing or invalid data.
Skipping row 2024-08-29 13:36:00 due to missing or invalid data.
Skipping row 2024-08-29 13:37:00 due to missing or invalid data.
Skipping row 2024-08-29 13:38:00 due to missing or invalid data.
Skipping row 2024-08-29 13:39:00 due to missing or invalid data.
Skipping row 2024-08-29 13:40:00 due to missing or invalid data.
Skipping row 2024-08-29 13:41:00 due to missing or invalid data.
Skipping row 2024-08-29 13:42:00 due to missing or invalid data.
Skipping row 2024-08-29 13:43:00 due to missing or invalid data.
Skipping row 2024-08-29 1

NameError: name 'trades' is not defined

In [ ]:
trade_details[0]

,timestamp,action,shares,inventory,time left
0,2024-08-28 09:30:00,"[0.1, 30.0]",1001,8999,470
1,2024-08-28 10:06:00,"[0.33299255, 35.40229]",2997,6002,434
2,2024-08-28 10:38:00,"[0.3395797, 31.996082]",2039,3963,402
3,2024-08-28 11:10:00,"[0.1, 31.162989]",397,3566,370
4,2024-08-28 11:40:00,"[0.33068854, 30.0]",1180,2386,340
5,2024-08-28 12:10:00,"[0.1, 30.0]",239,2147,310
6,2024-08-28 12:47:00,"[0.32034308, 36.86751]",688,1459,273
7,2024-08-28 13:17:00,"[0.18432893, 30.0]",269,1190,243
8,2024-08-28 13:51:00,"[0.1, 33.937317]",120,1070,209
9,2024-08-28 14:21:00,"[0.3415313, 30.0]",366,704,179
